In [0]:
%run /Users/rahulpatel@cyntexa.com/IAP_Practice/freshkart/00_set_up

In [0]:
catalog = dbutils.widgets.get("catalog")

In [0]:
# Source and checkpoint paths using the catalog parameter
source_path = f"/Volumes/{catalog}/landing/raw/stores/"
checkpoint_path = f"/Volumes/{catalog}/ops/pipeline_state/store_checkpoints/"
bronze_table = f"{catalog}.bronze.stores"

# Load CSV files using Auto Loader
raw_stores_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", f"{checkpoint_path}schema")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(source_path)
)

# Add metadata columns using the function from 00_set_up
bronze_stores_df = add_metadata(raw_stores_df)

# Write to bronze table (process all available files, then stop)
query = (
    bronze_stores_df.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(bronze_table)
)

query.awaitTermination()
print(f"Stores data loaded into {bronze_table}")

In [0]:
# --- Customers (arrival pattern: one full snapshot, then change files) ---
source_path = f"/Volumes/{catalog}/landing/raw/customers/"
bronze_table = f"{catalog}.bronze.customers"

# COPY INTO loads all available CSV files into the bronze table.
# Re-running this cell picks up new files since the last run (COPY INTO tracks file state).
spark.sql(f"""
    COPY INTO {bronze_table}
    FROM '{source_path}'
    FILEFORMAT = CSV
    FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'true')
    COPY_OPTIONS ('mergeSchema' = 'true')
""")

print(f"Customers data loaded into {bronze_table}")

In [0]:
%sql
-- select count(*) from fk_t12.bronze.customers;

In [0]:
# --- Orders (arrival pattern: one file per day, files can repeat/update earlier orders) ---
source_path = f"/Volumes/{catalog}/landing/raw/orders/"
checkpoint_path = f"/Volumes/{catalog}/ops/pipeline_state/order_checkpoints/"
bronze_table = f"{catalog}.bronze.orders"

# Auto Loader reads JSON lines files incrementally as they arrive daily.
# Some orders reappear in later files with updated status — bronze keeps all versions.
# Silver layer will deduplicate by order_id keeping the latest updated_at.
raw_orders_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", f"{checkpoint_path}schema")
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .load(source_path)
)

# Add metadata columns using the function from 00_set_up
bronze_orders_df = add_metadata(raw_orders_df)

# Write to bronze table (process all available files, then stop)
query = (
    bronze_orders_df.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(bronze_table)
)

query.awaitTermination()
print(f"Orders data loaded into {bronze_table}")

In [0]:
# --- Products (arrival pattern: full snapshot each week, multiline JSON) ---
source_path = f"/Volumes/{catalog}/landing/raw/products/"
checkpoint_path = f"/Volumes/{catalog}/ops/pipeline_state/products_checkpoints/"
bronze_table = f"{catalog}.bronze.products"

# Auto Loader reads multiline JSON (each file is a JSON array = full product snapshot).
# multiLine=true treats each file as one JSON document, expanding array elements to rows.
# Bronze appends every weekly snapshot; silver deduplicates by product_id keeping latest snapshot_date.
raw_products_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", f"{checkpoint_path}schema")
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .option("multiLine", "true")
    .load(source_path)
)

# Add metadata columns using the function from 00_set_up
bronze_products_df = add_metadata(raw_products_df)

# Write to bronze table (process all available files, then stop)
query = (
    bronze_products_df.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(bronze_table)
)

query.awaitTermination()
print(f"Products data loaded into {bronze_table}")